# Medical Insurance Charges — Exploratory Data Analysis

## Project Overview
This project explores a medical insurance dataset to understand how variables such as age, BMI, number of children, smoking status, sex, and region relate to insurance charges.

## Objectives
- Inspect the dataset and summarize its variables.
- Explore numeric distributions and potential outliers.
- Check missing values and duplicate records.
- Clean and encode categorical variables.
- Create BMI categories and examine associations with insurance charges.
- Prepare a selected feature table for further analysis.

## Tools
Python · Pandas · NumPy · Matplotlib · Seaborn · SciPy · scikit-learn

## Dataset
The notebook expects `insurance.csv` in the same folder.

> This is exploratory analysis, not medical or financial advice. Associations do not establish causation.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


## 2. Load Dataset

In [ ]:
df = pd.read_csv("insurance.csv")

## 3. Initial Data Inspection
Preview rows, inspect data types and descriptive statistics, and check missing values and columns.

In [ ]:
df

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

## 4. Exploratory Data Analysis
Explore numerical distributions, categorical counts, potential outliers, and correlations.

In [ ]:
numeric_columns =['age',  'bmi',  "children", 'charges']
for col in numeric_columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col],kde=True,bins= 20)

In [ ]:
sns.countplot(x = df["children"])

In [ ]:
sns.countplot(x=df['sex'])

In [ ]:
sns.countplot(x=df['smoker'])

In [ ]:
for col in numeric_columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[col])

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True),annot=True)

## 5. Data Cleaning and Preprocessing
Create a working copy, remove duplicate rows, inspect data quality, and encode categorical variables. The original `df` remains unchanged.

In [ ]:
df_cleaned = df.copy()

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.shape

In [ ]:
df_cleaned.drop_duplicates(inplace= True)

In [ ]:
df_cleaned.shape

In [ ]:
df_cleaned.isnull().sum()

In [ ]:
df_cleaned.dtypes

In [ ]:
df_cleaned['sex'].value_counts()

In [ ]:
df_cleaned["sex"] = df_cleaned['sex'].map({"male": 0,"female":1})

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned['smoker'].value_counts()

In [ ]:
df_cleaned['smoker'] = df_cleaned['smoker'].map({"yes":1,"no":0})

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.rename(columns={
    'sex':'is_female',
    'smoker':'is_smoker'
},inplace=True)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned['region'].value_counts()

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned,columns=["region"],drop_first=True)

In [ ]:
df_cleaned.head()

### Review Encoded Data Types
Inspect indicator columns and data types. Avoid converting standardized continuous values to integers because that removes decimal precision.

In [ ]:
df_cleaned.dtypes

## 6. Feature Engineering
Create BMI categories for descriptive analysis. These categories are not a medical diagnosis.

In [ ]:
sns.histplot(x=df_cleaned['bmi'])

In [ ]:
df_cleaned['bmi_catagory'] = pd.cut(
    df_cleaned['bmi'],
    bins= [0,18.5,24.9,29.9,float('inf')],
    labels=['underweight','normal','overweight','obese']
)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned,columns=['bmi_catagory'],drop_first=True)

In [ ]:
df_cleaned.head()

In [ ]:
# Keep continuous values as floats; inspect the engineered data.
df_cleaned.head()

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.columns

## 7. Feature Scaling
Standardize `age`, `bmi`, and `children`. For predictive modeling, fit the scaler on training data only to avoid data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

cols = ["age", "bmi", "children"]
scaler = StandardScaler()
df_cleaned[cols] = scaler.fit_transform(df_cleaned[cols])


In [ ]:
df_cleaned.head()


## 8. Feature Association Analysis
Calculate Pearson correlations between selected features and insurance charges. Correlation describes association, not causation.

In [ ]:
from scipy.stats import pearsonr

selected_features = ['age', 'is_female', 'bmi', 'children', 'is_smoker', 
       'region_northwest', 'region_southeast', 'region_southwest',
       'bmi_catagory_normal', 'bmi_catagory_overweight', 'bmi_catagory_obese']

correlation = {
    feature: pearsonr(df_cleaned[feature],df_cleaned["charges"])[0]
    for feature in selected_features
}
correlation_df = pd.DataFrame(list(correlation.items()),columns=['features','Pearson Correlation'])
correlation_df.sort_values(by='Pearson Correlation',ascending = False)

### Chi-Square Tests for Categorical Features
Charges are divided into quartiles and tested for association with categorical features. A small p-value is evidence against independence; it does not automatically determine whether a feature should be kept or dropped.

In [ ]:
cat_features = [ 'is_female', 'is_smoker', 
       'region_northwest', 'region_southeast', 'region_southwest',
       'bmi_catagory_normal', 'bmi_catagory_overweight', 'bmi_catagory_obese'
]


In [ ]:
from scipy.stats import chi2_contingency

alpha = 0.05
df_cleaned["charges_bin"] = pd.qcut(df_cleaned["charges"], q=4, labels=False, duplicates="drop")
chi2_results = {}

for col in cat_features:
    contingency = pd.crosstab(df_cleaned[col], df_cleaned["charges_bin"])
    chi2_stat, p_val, _, _ = chi2_contingency(contingency)
    decision = "Evidence of association" if p_val < alpha else "Insufficient evidence of association"
    chi2_results[col] = {
        "chi2_statistic": chi2_stat,
        "p_value": p_val,
        "Interpretation": decision
    }

chi2_df = pd.DataFrame(chi2_results).T.sort_values(by="p_value")
chi2_df


## 9. Selected Analysis Dataset
Select a subset of columns for possible follow-up analysis. This is not a trained or validated prediction model.

In [ ]:
final_df = df_cleaned[['age','is_female','bmi','children','is_smoker','charges','region_southeast','bmi_catagory_obese']]

In [ ]:
final_df

## 10. Key Takeaways and Limitations
- The notebook explores how demographic, lifestyle, and geographic variables relate to insurance charges.
- Visualizations, correlations, and chi-square tests can help identify patterns worth investigating.
- Statistical association is not proof of causation.
- Review the selected columns before using them for any predictive task.
- For predictive modeling, use a train/test split and fit preprocessing steps on training data only.
